# Props Data Organizer

## Definitions of Formulas Used in This Notebook

### Estimated Value (EV)
EV is the average amount you can expect to win or lose per bet if you placed the same bet many times. It helps identify profitable betting opportunities by comparing the expected return to the risk involved.

**Formula:**
$$
\text{EV} = (\text{Probability of Winning} \times \text{Profit if Win}) - (\text{Probability of Losing} \times \text{Loss if Lose})
$$

### Kelly Criterion
The Kelly Criterion is a formula used to determine the optimal size of a series of bets. It aims to maximize the logarithm of wealth, balancing the trade-off between risk and reward. The formula considers both the probability of winning and the odds offered, guiding you on how much of your bankroll to wager on each bet.

**Formula:**
$$
\text{Kelly Fraction} = \frac{(\text{Probability of Winning} \times (\text{Odds} + 1)) - 1}{\text{Odds}}
$$

### Variance
Variance in sports betting represents the spread or dispersion of actual outcomes around the expected value. It's a crucial metric for understanding the risk and volatility associated with betting predictions. Higher variance indicates more volatile and unpredictable outcomes, while lower variance suggests more consistent results.

**Formula:**
$$
\text{Variance} = \frac{\sum_{i=1}^{n} (x_i - \mu)^2}{n}
$$

Where:
- $x_i$ represents each individual outcome
- $\mu$ is the mean or expected value
- $n$ is the total number of observations

In the context of prop betting:
- High variance props (e.g., 3-pointers made) tend to be more risky but potentially more profitable
- Low variance props (e.g., minutes played) typically offer more consistent but lower returns


In [2]:
import pandas as pd 
import numpy as np
import time
import requests
import os
import sys
from datetime import datetime
import joblib

feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
# feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)
    
from PROPS_EV.calculateEVS import *
from MODELS.model import *

today = datetime.now()
formatted_date = today.strftime("%m_%d_%y")

c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\venv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Grabs players odds for the day (US all boookmakers, DFS is prizepicks and underdogs)

In [2]:
# from NBAPropFinder.NBAPropFinder import NBAPropFinder

# nba_props = NBAPropFinder(region='us_dfs')
# prizePicks = nba_props.dataframe
# prizePicks.head(10)

### Single Bets from bookmakers that dont include prizePicks or UnderDogs

In [15]:
features = [
    # Player context
    'PLAYER_ID', 'TEAM_ID', 'OPP_TEAM_ID', 
    'STARTING', 'HOME_GAME', 
    'PLAYER_DAYS_REST', 'IS_BACK_TO_BACK', 
    
    # Star Players
    # 'PLAYER_IS_TEAM_STAR', 'TEAM_STAR_OUT',
    # 'PTS_WITHOUT_STAR', 'MIN_WITHOUT_STAR', 'USG_PCT_WITHOUT_STAR', 'FGA_WITHOUT_STAR', 'FG3A_WITHOUT_STAR', 'FTA_WITHOUT_STAR', 
    # 'EFG_PCT_WITHOUT_STAR', 'TS_PCT_WITHOUT_STAR', 'AST_WITHOUT_STAR', 'REB_WITHOUT_STAR', 'PTS_PER_36_WITHOUT_STAR',
    
    # Player season averages
    'MIN_SEASON_AVG_TO_DATE', 'PTS_SEASON_AVG_TO_DATE','FGA_SEASON_AVG_TO_DATE','FG3A_SEASON_AVG_TO_DATE',
    'FTA_SEASON_AVG_TO_DATE','USG_PCT_SEASON_AVG_TO_DATE','TS_PCT_SEASON_AVG_TO_DATE',
    'EFG_PCT_SEASON_AVG_TO_DATE', 'AST_SEASON_AVG_TO_DATE', 'REB_SEASON_AVG_TO_DATE', 'TOV_SEASON_AVG_TO_DATE',
    
    # LAG
    'PTS_LAG_1', 'PTS_LAG_2',
    'FGA_LAG_1', 'FGA_LAG_2',
    'MIN_LAG_1', 'MIN_LAG_2',
    'USG_PCT_LAG_1', 'USG_PCT_LAG_2',
    
    # Short-term form (5-game rolling averages)
    'MIN_ROLLING_AVG_5', 'PTS_ROLLING_AVG_5', 'FGA_ROLLING_AVG_5',
    'FG3A_ROLLING_AVG_5', 'FTA_ROLLING_AVG_5', 'USG_PCT_ROLLING_AVG_5',
    'TS_PCT_ROLLING_AVG_5','EFG_PCT_ROLLING_AVG_5', 'AST_ROLLING_AVG_5', 
    'REB_ROLLING_AVG_5', 'TOV_ROLLING_AVG_5',
    
    # Medium-term form (15-game rolling averages)
    'MIN_ROLLING_AVG_15', 'PTS_ROLLING_AVG_15', 'FGA_ROLLING_AVG_15',
    'FG3A_ROLLING_AVG_15', 'FTA_ROLLING_AVG_15', 'USG_PCT_ROLLING_AVG_15',
    'TS_PCT_ROLLING_AVG_15','EFG_PCT_ROLLING_AVG_15', 'AST_ROLLING_AVG_15', 
    'REB_ROLLING_AVG_15', 'TOV_ROLLING_AVG_15',
    
    # Long-term form (40-game rolling averages)
    'MIN_ROLLING_AVG_40', 'PTS_ROLLING_AVG_40', 'FGA_ROLLING_AVG_40', 'FG3A_ROLLING_AVG_40', 'FTA_ROLLING_AVG_40',
    'USG_PCT_ROLLING_AVG_40', 'TS_PCT_ROLLING_AVG_40', 'EFG_PCT_ROLLING_AVG_40', 'AST_ROLLING_AVG_40', 
    'REB_ROLLING_AVG_40', 'TOV_ROLLING_AVG_40',

    # Opponent
    'OPP_DEF_RATING_AVG_TO_DATE', 'OPP_PACE_AVG_TO_DATE', 'OPP_PTS_AVG_TO_DATE', 'OPP_FGA_AVG_TO_DATE', 
    'OPP_REB_AVG_TO_DATE', 'OPP_AST_AVG_TO_DATE', 'OPP_TOV_AVG_TO_DATE', 'OPP_BLK_AVG_TO_DATE', 'OPP_STL_AVG_TO_DATE',
    
    #starters 
    'TEAM_OFF_RATING_AVG_TO_DATE','TEAM_DEF_RATING_AVG_TO_DATE','TEAM_PACE_AVG_TO_DATE', 'TEAM_FGA_AVG_TO_DATE',
    'TEAM_PTS_AVG_TO_DATE', 'TEAM_REB_AVG_TO_DATE', 'TEAM_AST_AVG_TO_DATE', 'TEAM_TOV_AVG_TO_DATE',

    # Matchup micro-feature
    'MATCHUP_AVG_MIN_LAST_3_TO_DATE', 'MATCHUP_AVG_FGA_LAST_3_TO_DATE', 'MATCHUP_AVG_FG3A_LAST_3_TO_DATE', 'MATCHUP_AVG_FTA_LAST_3_TO_DATE', 
    'MATCHUP_AVG_PTS_LAST_3_TO_DATE', 'MATCHUP_AVG_USG_PCT_LAST_3_TO_DATE', 'MATCHUP_AVG_EFG_PCT_LAST_3_TO_DATE', 'MATCHUP_AVG_TS_PCT_LAST_3_TO_DATE', 
    'MATCHUP_AVG_AST_LAST_3_TO_DATE', 'MATCHUP_AVG_REB_LAST_3_TO_DATE', 'MATCHUP_AVG_TOV_LAST_3_TO_DATE',
    
    # Team odds
    'team_spread', 'total', 'team_is_favored','TEAM_IMPLIED_PTS_FAV','TEAM_IMPLIED_PTS_UND','BLOWOUT_RISK'
]

# model = joblib.load('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/Models/PTS_cat_model.pkl')
model = joblib.load(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\MODELS\Models\PTS_cat_model.pkl")
data = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values('GAME_DATE', ascending=False)
bookmakers = pd.read_csv('../BACKTESTS/odds25.csv')
odds = bookmakers[(bookmakers['CATEGORY'] == 'player_points') & (bookmakers['GAME_DATE'] == '2025-04-11') & (bookmakers['BOOKMAKER'] == 'prizepicks')]
games = get_espn_games(date_str='20250411')
odds

C:\Users\alexg\AppData\Local\Temp\ipykernel_22060\1873344069.py:60: DtypeWarning: Columns (10,11,13) have mixed types. Specify dtype option on import or set low_memory=False.
  bookmakers = pd.read_csv('../BACKTESTS/odds25.csv')


,Unnamed: 0.1,Unnamed: 0,NAME,CATEGORY,BOOKMAKER,OVER/UNDER,LINE,PRICE,HOME_TEAM,AWAY_TEAM,game_id,commence_time,GAME_DATE,period_id,fair_line,fair_odds,OVER/ODDS
191672,191672,111661,Kyle Kuzma,player_points,prizepicks,over,12.5,100,Milwaukee Bucks,New Orleans Pelicans,NaN,NaN,2025-04-11,game,17.5,213.0,over
191676,191676,111665,Kyle Kuzma,player_points,prizepicks,under,12.5,100,Milwaukee Bucks,New Orleans Pelicans,NaN,NaN,2025-04-11,game,17.5,-213.0,under
191679,191679,111668,Ryan Rollins,player_points,prizepicks,under,9.5,100,Milwaukee Bucks,New Orleans Pelicans,NaN,NaN,2025-04-11,game,9.5,104.0,under
191682,191682,111671,Ryan Rollins,player_points,prizepicks,over,9.5,100,Milwaukee Bucks,New Orleans Pelicans,NaN,NaN,2025-04-11,game,9.5,-104.0,over
191687,191687,111676,Taurean Prince,player_points,prizepicks,over,7.5,100,Milwaukee Bucks,New Orleans Pelicans,NaN,NaN,2025-04-11,game,6.5,118.0,over
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193173,193173,113512,Jarrett Allen,player_points,prizepicks,under,8.5,100,New York Knicks,Cleveland Cavaliers,NaN,NaN,2025-04-11,game,8.5,-103.0,under
193181,193181,113520,Isaac Okoro,player_points,prizepicks,over,10.5,100,New York Knicks,Cleveland Cavaliers,NaN,NaN,2025-04-11,game,10.5,113.0,over
193185,193185,113524,Isaac Okoro,player_points,prizepicks,under,10.5,100,New York Knicks,Cleveland Cavaliers,NaN,NaN,2025-04-11,game,10.5,-113.0,under
193189,193189,113528,Evan Mobley,player_points,prizepicks,over,16.0,100,New York Knicks,Cleveland Cavaliers,NaN,NaN,2025-04-11,game,15.5,104.0,over


In [16]:
filterData = data[data['GAME_DATE'] <= '2025-04-11'].sort_values('GAME_DATE', ascending=True)
filterData

,Unnamed: 0,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,...,TEAM_TOV_AVG_TO_DATE,OPP_DEF_RATING_AVG_TO_DATE,OPP_PACE_AVG_TO_DATE,OPP_PTS_AVG_TO_DATE,OPP_FGA_AVG_TO_DATE,OPP_REB_AVG_TO_DATE,OPP_AST_AVG_TO_DATE,OPP_TOV_AVG_TO_DATE,OPP_BLK_AVG_TO_DATE,OPP_STL_AVG_TO_DATE
0,0,LeBron James,2544,LAL vs. MIN,LAL,1610612747,MIN,1,22400062,2024-10-22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16629,16629,Sam Hauser,1630573,BOS vs. NYK,BOS,1610612738,NYK,1,22400061,2024-10-22,...,11.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14204,14204,Xavier Tillman,1630214,BOS vs. NYK,BOS,1610612738,NYK,1,22400061,2024-10-22,...,11.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24251,24251,Dalton Knecht,1642261,LAL vs. MIN,LAL,1610612747,MIN,1,22400062,2024-10-22,...,13.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16700,16700,Ariel Hukporti,1630574,NYK @ BOS,NYK,1610612752,BOS,0,22400061,2024-10-22,...,13.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16698,16698,Sam Hauser,1630573,BOS vs. CHA,BOS,1610612738,CHA,1,22401174,2025-04-11,...,11.81,113.04,100.40,105.48,89.12,45.09,24.32,15.46,4.50,7.38
22483,22483,Adem Bona,1641737,PHI vs. ATL,PHI,1610612755,ATL,1,22401173,2025-04-11,...,13.62,113.10,105.36,118.12,91.82,44.40,29.58,15.52,5.15,9.76
9946,9946,Luka Dončić,1629029,LAL vs. HOU,LAL,1610612747,HOU,1,22401185,2025-04-11,...,14.13,106.60,101.55,114.40,93.36,48.65,23.25,13.88,5.03,8.45
22380,22380,Nick Smith Jr.,1641733,CHA @ BOS,CHA,1610612766,BOS,0,22401174,2025-04-11,...,15.43,108.89,98.37,116.39,89.96,45.19,26.09,11.88,5.48,7.10


## Best EVs for Single Bets from draftkings, fanduel, prizepicks, and underdog

In [17]:
final_results = single_bet(filterData, odds, model, games, features, '20250411', stake=10, simulations=20000).sort_values(by='EV%', ascending=False).reset_index(drop=True)

print("\nTop 10 highest EV bets across point props:")
final_results.head(10)

Processing single bets...

Top 10 highest EV bets across point props:


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,OVER%,UNDER%,EV$,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
0,Quentin Grimes,prizepicks,player_points,24.5,100,under,17.75,0.014,0.986,9.71,0.97,0.97,0.49,0.24,"(11.8, 23.8)"
1,Karlo Matković,prizepicks,player_points,14.5,100,under,10.72,0.028,0.972,9.44,0.94,0.94,0.47,0.24,"(6.8, 14.6)"
2,Adem Bona,prizepicks,player_points,13.5,100,under,10.74,0.085,0.915,8.31,0.83,0.83,0.42,0.21,"(6.9, 14.7)"
3,Goga Bitadze,prizepicks,player_points,12.5,100,under,7.24,0.134,0.866,7.32,0.73,0.73,0.37,0.18,"(0.8, 16.3)"
4,Antonio Reeves,prizepicks,player_points,19.5,100,under,10.57,0.203,0.797,5.95,0.59,0.59,0.30,0.15,"(0.9, 29.5)"
5,Jarrett Allen,prizepicks,player_points,8.5,100,over,12.60,0.781,0.219,5.61,0.56,0.56,0.28,0.14,"(2.7, 23.3)"
6,Lonnie Walker IV,prizepicks,player_points,20.0,100,under,14.14,0.249,0.751,5.02,0.50,0.50,0.25,0.13,"(1.8, 30.4)"
7,Obi Toppin,prizepicks,player_points,15.5,100,under,11.38,0.254,0.746,4.92,0.49,0.49,0.25,0.12,"(1.7, 23.1)"
8,Donte DiVincenzo,prizepicks,player_points,11.5,100,under,9.11,0.283,0.717,4.35,0.43,0.43,0.22,0.11,"(1.9, 17.1)"
9,Isaac Okoro,prizepicks,player_points,10.5,100,under,7.50,0.296,0.704,4.09,0.41,0.41,0.20,0.10,"(0.9, 17.5)"


In [22]:
from datetime import datetime
import numpy as np
import pandas as pd

from MODELS.pipeline import buildFeatureVector, makePredictionCatBoost, getStarters, findOppTeam
from PROPS_EV.calculateEVS import monteCarloSim

def prizepickspairsEV(data, bookmakers, model, gamesSchedule, features, todayDate, stake=100,
                      simulations=10000, std_window=10, min_std=2.0, max_std=9.5, stat_col='PTS'):
    print("Processing PrizePicks pairs...")
    date_obj = datetime.strptime(todayDate, "%Y%m%d")
    _game_date = date_obj.strftime("%Y-%m-%d")
    todayDate = str(todayDate)

    def get_player_std(player_df, stat_col):
        s = player_df[stat_col].dropna()
        if s.empty:
            return 5.0
        sd = s.tail(std_window).std(ddof=1) if len(s) >= std_window else s.std(ddof=1)
        if pd.isna(sd) or sd == 0:
            sd = 5.0
        return float(np.clip(sd, min_std, max_std))

    legs = []
    for _, row in bookmakers.iterrows():
        name = row['NAME']
        category = row['CATEGORY']
        line = float(row['LINE'])
        side = row.get('OVER/UNDER', 'over')

        player_df = data[data['PLAYER_NAME'] == name].sort_values(by='GAME_DATE', ascending=False)
        if player_df.empty or stat_col not in player_df.columns:
            continue

        player_team = player_df['TEAM_ABBREVIATION'].iloc[0]
        game_id = int(player_df['GAME_ID'].iloc[0])

        starters = getStarters(game_id, player_team, data)
        opponent, _ = findOppTeam(name, data, gamesSchedule)
        if opponent is None:
            continue

        temp_props = pd.DataFrame({
            'player': [name],
            'line': [line],
            'NAME': [name],
            'LINE': [line],
            'CATEGORY': [category]
        })

        features_list = features
        if features_list is None:
            fv = buildFeatureVector(name, data, gamesSchedule, todayDate, starters, game_id)
            features_list = [f'f{i}' for i in range(len(fv))]

        pred = makePredictionCatBoost(
            player_name=name,
            data=data,
            model=model,
            bookmakers=temp_props,
            games=gamesSchedule,
            todayDate=todayDate,
            starters=starters,
            game_id=game_id,
            features=features_list,
            n_games=3
        )

        std_dev = get_player_std(player_df, stat_col)

        sim_results = monteCarloSim(
            player_df=player_df,
            modelPred=float(pred['predicted_stat']),
            prop_line=line,
            std_dev=std_dev,
            num_simulations=simulations
        )

        legs.append({
            'NAME': name,
            'CATEGORY': category,
            'LINE': line,
            'SIDE': side,
            'PREDICTION': float(pred['predicted_stat']),
            'OVER%': float(sim_results['prob_over']),
            'UNDER%': float(sim_results['prob_under']),
            'CI': sim_results['confidence_interval']
        })

    payout_multiple = 3.0
    b = payout_multiple - 1.0

    pair_results = []
    for i in range(len(legs)):
        for j in range(i + 1, len(legs)):  # i + 1 prevents duplicate combos (a,b vs b,a)
            leg1 = legs[i]
            leg2 = legs[j]
            
            # Skip if same player (no duplicates)
            if leg1['NAME'] == leg2['NAME']:
                continue
            
            # Skip if players from same team
            player1_team = data[data['PLAYER_NAME'] == leg1['NAME']]['TEAM_ABBREVIATION'].iloc[0]
            player2_team = data[data['PLAYER_NAME'] == leg2['NAME']]['TEAM_ABBREVIATION'].iloc[0]
            if player1_team == player2_team:
                continue

            p1 = leg1['OVER%'] if str(leg1['SIDE']).upper().startswith('O') else leg1['UNDER%']
            p2 = leg2['OVER%'] if str(leg2['SIDE']).upper().startswith('O') else leg2['UNDER%']
            p_both = p1 * p2

            ev_per_unit = p_both * b - (1 - p_both)
            ev_dollars = stake * ev_per_unit
            ev_percent = ev_per_unit

            kelly_full = max(0.0, (b * p_both - (1 - p_both)) / b) if b > 0 else 0.0

            pair_results.append({
                'PLAYER 1': leg1['NAME'],
                'CATEGORY 1': leg1['CATEGORY'],
                'LINE 1': leg1['LINE'],
                'SIDE 1': leg1['SIDE'],
                'PREDICTION 1': round(leg1['PREDICTION'], 2),
                'OVER% 1': round(leg1['OVER%'], 3),
                'UNDER% 1': round(leg1['UNDER%'], 3),
                'CONFIDENCE INTERVAL 1': f"({leg1['CI'][0]:.1f}, {leg1['CI'][1]:.1f})",
                'PLAYER 2': leg2['NAME'],
                'CATEGORY 2': leg2['CATEGORY'],
                'LINE 2': leg2['LINE'],
                'SIDE 2': leg2['SIDE'],
                'PREDICTION 2': round(leg2['PREDICTION'], 2),
                'OVER% 2': round(leg2['OVER%'], 3),
                'UNDER% 2': round(leg2['UNDER%'], 3),
                'CONFIDENCE INTERVAL 2': f"({leg2['CI'][0]:.1f}, {leg2['CI'][1]:.1f})",
                'TYPE': f"{'OVER' if str(leg1['SIDE']).upper().startswith('O') else 'UNDER'}/"
                        f"{'OVER' if str(leg2['SIDE']).upper().startswith('O') else 'UNDER'}",
                'PROBABILITY': round(p_both, 4),
                'EV$': round(ev_dollars, 2),
                'EV%': round(ev_percent, 3),
                'KELLY FULL': round(kelly_full, 2),
                'KELLY HALF': round(0.5 * kelly_full, 2),
                'KELLY QUARTER': round(0.25 * kelly_full, 2)
            })

    return pd.DataFrame(pair_results)
    
# Use the custom function we just created
result = prizepickspairsEV(filterData, odds, model, games, features, '20250411')

Processing PrizePicks pairs...


In [23]:
result.sort_values(by='EV%', ascending=False)

,PLAYER 1,CATEGORY 1,LINE 1,SIDE 1,PREDICTION 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,...,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,TYPE,PROBABILITY,EV$,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER
1999,Karlo Matković,player_points,14.5,under,10.72,0.028,0.972,"(6.8, 14.6)",Quentin Grimes,player_points,...,0.016,0.984,"(11.8, 23.9)",UNDER/UNDER,0.9567,187.02,1.870,0.94,0.47,0.23
5789,Quentin Grimes,player_points,24.5,under,17.75,0.016,0.984,"(11.8, 23.9)",Adem Bona,player_points,...,0.080,0.920,"(6.8, 14.6)",UNDER/UNDER,0.9051,171.53,1.715,0.86,0.43,0.21
2001,Karlo Matković,player_points,14.5,under,10.72,0.028,0.972,"(6.8, 14.6)",Adem Bona,player_points,...,0.080,0.920,"(6.8, 14.6)",UNDER/UNDER,0.8945,168.35,1.684,0.84,0.42,0.21
5815,Quentin Grimes,player_points,24.5,under,17.75,0.016,0.984,"(11.8, 23.9)",Goga Bitadze,player_points,...,0.137,0.863,"(0.8, 16.4)",UNDER/UNDER,0.8496,154.88,1.549,0.77,0.39,0.19
2027,Karlo Matković,player_points,14.5,under,10.72,0.028,0.972,"(6.8, 14.6)",Goga Bitadze,player_points,...,0.137,0.863,"(0.8, 16.4)",UNDER/UNDER,0.8397,151.90,1.519,0.76,0.38,0.19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1526,Antonio Reeves,player_points,19.5,over,10.57,0.201,0.799,"(0.9, 29.3)",Quentin Grimes,player_points,...,0.015,0.985,"(11.7, 23.9)",OVER/OVER,0.0030,-99.11,-0.991,0.00,0.00,0.00
1884,Karlo Matković,player_points,14.5,over,10.72,0.031,0.969,"(6.9, 14.7)",Adem Bona,player_points,...,0.087,0.913,"(6.8, 14.6)",OVER/OVER,0.0027,-99.19,-0.992,0.00,0.00,0.00
5738,Quentin Grimes,player_points,24.5,over,17.75,0.015,0.985,"(11.7, 23.9)",Goga Bitadze,player_points,...,0.138,0.862,"(0.8, 16.3)",OVER/OVER,0.0020,-99.39,-0.994,0.00,0.00,0.00
5712,Quentin Grimes,player_points,24.5,over,17.75,0.015,0.985,"(11.7, 23.9)",Adem Bona,player_points,...,0.087,0.913,"(6.8, 14.6)",OVER/OVER,0.0013,-99.61,-0.996,0.00,0.00,0.00


In [ ]:
# from NBAData.backtest import PrizePicksBacktest
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Tuple, NamedTuple
from dataclasses import dataclass

@dataclass
class BetResult:
    date: str
    player1: str
    category1: str
    line1: float
    actual1: float
    player2: str
    category2: str
    line2: float
    actual2: float
    bet_type: str
    ev: float
    probability: float
    kelly: float
    won: bool
    profit: float

class PrizePicksBacktest:
    def __init__(self, 
                 props_ev_dir: str = "CSV_FILES/HISTORICAL_PROP_PAIRS",
                 regular_data_dir: str = "CSV_FILES/REGULAR_DATA",
                 min_ev: float = 60.0,
                 stake: float = 100,
                 max_bets_per_day: int = 3,
                 kelly_fraction: float = 0.25):
        """
        Initialize backtester for PrizePicks pairs using actual results
        
        Args:
            props_ev_dir: Directory containing PrizePicks EV CSV files
            regular_data_dir: Directory containing actual game results
            min_ev: Minimum EV threshold for taking a bet
            stake: Stake size for each bet
        """
        self.props_ev_dir = Path(props_ev_dir)
        self.regular_data_dir = Path(regular_data_dir)
        self.min_ev = min_ev
        self.max_bets_per_day = max_bets_per_day
        self.stake = stake
        self.kelly_fraction = kelly_fraction
        self.results: List[BetResult] = []
        self.bet_selection_log = []  # Track bet selection process
        
        # Load actual results data
        self.actual_results = self._load_actual_results()
        
    def _load_actual_results(self) -> Dict[str, pd.DataFrame]:
        """Load actual results for each stat category"""
        results = {}
        stat_types = {
            'player_points': 'PTS',
            'player_rebounds': 'REB',
            'player_assists': 'AST'
        }
        
        for category, stat in stat_types.items():
            file_path = self.regular_data_dir / f'season_25_{stat}_features.csv'
            if file_path.exists():
                df = pd.read_csv(file_path)
                # Convert date to YYYYMMDD format
                df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE']).dt.strftime('%Y%m%d')
                results[category] = df
                
        return results
    
    def _get_actual_stat(self, date: str, player: str, category: str) -> float:
        """Get actual stat value for a player on a given date"""
        if category not in self.actual_results:
            return None
            
        df = self.actual_results[category]
        result = df[(df['GAME_DATE'] == date) & (df['PLAYER_NAME'] == player)]
        
        if result.empty:
            return None
            
        stat_map = {
            'player_points': 'PTS',
            'player_rebounds': 'REB',
            'player_assists': 'AST'
        }
        
        return result[stat_map[category]].iloc[0]

    def _check_bet_result(self, bet_type: str, actual1: float, line1: float, 
                     actual2: float, line2: float) -> bool:
        """Check if a bet won based on actual results"""
        if actual1 is None or actual2 is None:
            return None
            
        bet_parts = bet_type.split('/')
        result1 = actual1 > line1 if bet_parts[0] == 'OVER' else actual1 < line1
        result2 = actual2 > line2 if bet_parts[1] == 'OVER' else actual2 < line2
        
        return result1 and result2

    def load_daily_ev_data(self, date_str: str) -> pd.DataFrame:
        """Load PrizePicks pairs EV data for a specific date"""
        file_path = self.props_ev_dir / f"{date_str}_PAIRS.csv"
        if not file_path.exists():
            return pd.DataFrame()
        
        df = pd.read_csv(file_path)
        # First filter by minimum EV
        qualified_bets = df[df['EV'] >= self.min_ev].sort_values('EV', ascending=False)
        # Then take only top N bets
        return qualified_bets.head(self.max_bets_per_day)

    def simulate_bets(self) -> None:
        """Run backtest simulation using actual results"""
        ev_files = list(self.props_ev_dir.glob("*_PAIRS.csv"))
        
        for file in sorted(ev_files):
            date_str = file.stem.split('_')[0]  # Get YYYYMMDD from filename
            
            # Load all bets for the day
            all_bets = pd.read_csv(file)
            daily_bets = self.load_daily_ev_data(date_str)
            
            # Log bet selection process
            self.bet_selection_log.append({
                'date': date_str,
                'total_available_bets': len(all_bets),
                'bets_above_min_ev': len(all_bets[all_bets['EV'] >= self.min_ev]),
                'bets_selected': len(daily_bets),
                'min_ev_selected': daily_bets['EV'].min() if not daily_bets.empty else None,
                'max_ev_selected': daily_bets['EV'].max() if not daily_bets.empty else None
            })
            
            if daily_bets.empty:
                continue
                
            for _, bet in daily_bets.iterrows():
                # Get actual results
                actual1 = self._get_actual_stat(date_str, bet['PLAYER 1'], bet['CATEGORY 1'])
                actual2 = self._get_actual_stat(date_str, bet['PLAYER 2'], bet['CATEGORY 2'])
                
                # Skip if we don't have actual results
                if actual1 is None or actual2 is None:
                    continue
                
                # Check if bet won
                won = self._check_bet_result(
                    bet['TYPE'], 
                    actual1, bet['PLAYER 1 LINE'],
                    actual2, bet['PLAYER 2 LINE']
                )
                
                if won is None:
                    continue
                    
                kelly_stake = self.stake * bet['KELLY CRITERION'] * self.kelly_fraction
                profit = kelly_stake if won else -kelly_stake
                
                result = BetResult(
                    date=date_str,
                    player1=bet['PLAYER 1'],
                    category1=bet['CATEGORY 1'],
                    line1=bet['PLAYER 1 LINE'],
                    actual1=actual1,
                    player2=bet['PLAYER 2'],
                    category2=bet['CATEGORY 2'],
                    line2=bet['PLAYER 2 LINE'],
                    actual2=actual2,
                    bet_type=bet['TYPE'],
                    ev=bet['EV'],
                    probability=bet['PROBABILITY'],
                    kelly=bet['KELLY CRITERION'],
                    won=won,
                    profit=profit
                )
                self.results.append(result)

    def calculate_metrics(self) -> Dict[str, float]:
        """Calculate performance metrics"""
        if not self.results:
            return {}
            
        profits = [r.profit for r in self.results]
        cumulative_profits = np.cumsum(profits)
        
        total_bets = len(self.results)
        winning_bets = sum(1 for r in self.results if r.won)
        total_risked = sum(abs(r.profit) for r in self.results)  # Sum of absolute profits since each profit represents the Kelly stake

        metrics = {
            'total_bets': total_bets,
            'hit_rate': winning_bets / total_bets if total_bets > 0 else 0,
            'total_profit': sum(profits),
            'roi': (sum(profits) / total_risked) if total_bets > 0 else 0,
            'max_drawdown': self._calculate_max_drawdown(cumulative_profits),
            'sharpe_ratio': self._calculate_sharpe_ratio(profits)
        }
        
        return metrics
    
    def _calculate_max_drawdown(self, cumulative_profits: np.ndarray) -> float:
        """Calculate maximum drawdown"""
        rolling_max = np.maximum.accumulate(cumulative_profits)
        drawdowns = rolling_max - cumulative_profits
        return np.max(drawdowns) if len(drawdowns) > 0 else 0
    
    def _calculate_sharpe_ratio(self, profits: List[float], risk_free_rate: float = 0.0) -> float:
        """Calculate Sharpe ratio"""
        if not profits:
            return 0.0
        
        returns = np.array(profits) / self.stake
        excess_returns = returns - risk_free_rate
        if len(excess_returns) < 2:
            return 0.0
            
        return np.mean(excess_returns) / np.std(excess_returns, ddof=1) if np.std(excess_returns, ddof=1) != 0 else 0.0

    def print_summary(self) -> None:
        """Print backtest summary with actual results"""
        metrics = self.calculate_metrics()
        if not metrics:
            print("No results to display")
            return
            
        print("\n=== PrizePicks Pairs Backtest Summary ===")
        print(f"Total Bets: {metrics['total_bets']}")
        print(f"Total Days: {len(self.bet_selection_log)}")
        print(f"Average Bets Per Day: {metrics['total_bets']/len(self.bet_selection_log):.1f}")
        print(f"Hit Rate: {metrics['hit_rate']:.2%}")
        
        # Calculate and display Kelly sizing statistics
        total_risked = sum(abs(r.profit) for r in self.results)
        avg_stake = total_risked / metrics['total_bets'] if metrics['total_bets'] > 0 else 0
        print(f"Total Amount Risked: ${total_risked:.2f}")
        print(f"Average Stake Size: ${avg_stake:.2f}")
        
        print(f"Total Profit: ${metrics['total_profit']:.2f}")
        print(f"ROI: {metrics['roi']:.2%}")
        print(f"Max Drawdown: ${metrics['max_drawdown']:.2f}")
        print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
        
        # Print bet selection statistics
        print("\nBet Selection Statistics:")
        total_available = sum(log['total_available_bets'] for log in self.bet_selection_log)
        total_above_min_ev = sum(log['bets_above_min_ev'] for log in self.bet_selection_log)
        print(f"Average Available Bets Per Day: {total_available/len(self.bet_selection_log):.1f}")
        print(f"Average Bets Above Min EV Per Day: {total_above_min_ev/len(self.bet_selection_log):.1f}")
        
        # Print top 5 highest EV bets and their actual results
        print("\nTop 5 Highest EV Bets:")
        top_ev_bets = sorted(self.results, key=lambda x: x.ev, reverse=True)[:5]
        for bet in top_ev_bets:
            print(f"{bet.date}: {bet.player1} {bet.category1} {bet.line1} (Actual: {bet.actual1:.1f}) & "
                  f"{bet.player2} {bet.category2} {bet.line2} (Actual: {bet.actual2:.1f})")
            print(f"Type: {bet.bet_type}, EV: {bet.ev:.2f}, Won: {bet.won}")

    def plot_performance(self) -> None:
        """Plot cumulative performance over time"""
        try:
            import matplotlib.pyplot as plt
            
            if not self.results:
                print("No results to plot")
                return
                
            profits = [r.profit for r in self.results]
            cumulative_profits = np.cumsum(profits)
            dates = [datetime.strptime(r.date, '%Y%m%d') for r in self.results]
            
            plt.figure(figsize=(12, 6))
            plt.plot(dates, cumulative_profits, label='Cumulative Profit')
            plt.axhline(y=0, color='r', linestyle='--', alpha=0.3)
            plt.title('PrizePicks Pairs Performance')
            plt.xlabel('Date')
            plt.ylabel('Profit ($)')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
            
        except ImportError:
            print("matplotlib is required for plotting")
# Initialize backtester
backtester = PrizePicksBacktest(min_ev=60.0, stake=100, max_bets_per_day=5, kelly_fraction=0.25)

# Run backtest with actual results; 
backtester.simulate_bets()

# Print results
backtester.print_summary()


=== PrizePicks Pairs Backtest Summary ===
Total Bets: 362
Total Days: 77
Average Bets Per Day: 4.7
Hit Rate: 40.06%
Total Amount Risked: $7269.00
Average Stake Size: $20.08
Total Profit: $-1361.75
ROI: -18.73%
Max Drawdown: $1703.91
Sharpe Ratio: -0.19

Bet Selection Statistics:
Average Available Bets Per Day: 5.0
Average Bets Above Min EV Per Day: 5.0

Top 5 Highest EV Bets:
20241118: Trey Lyles player_points 13.5 (Actual: 12.0) & Kevin Huerter player_rebounds 5.0 (Actual: 5.0)
Type: UNDER/UNDER, EV: 189.53, Won: False
20241231: Mason Plumlee player_rebounds 10.5 (Actual: 8.0) & John Konchar player_assists 2.5 (Actual: 4.0)
Type: UNDER/UNDER, EV: 186.41, Won: False
20241231: John Konchar player_points 6.5 (Actual: 7.0) & Mason Plumlee player_rebounds 10.5 (Actual: 8.0)
Type: UNDER/UNDER, EV: 184.05, Won: False
20241118: Trey Lyles player_points 13.5 (Actual: 12.0) & Kevin Huerter player_points 16.0 (Actual: 9.0)
Type: UNDER/UNDER, EV: 183.43, Won: True
20241025: Andre Drummond player